In [1]:
import time
import numpy as np
import xesmf as xe
import xarray as xr
from pathlib import Path
from dask.distributed import Client
from dask.diagnostics import ProgressBar

import warnings
warnings.filterwarnings("ignore")

In [2]:
c = Client(threads_per_worker=1)

2026-04-21 11:00:46,349 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/nm03/lxy581/evaluate/amp_phase/form_vel_amp_pha_test_013.nc', lease_id='c545ec2dfdcb4cd78b3da2e1061f9a6b'. This can happen if the Lock or Semaphore timed out before.
2026-04-21 11:04:58,816 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/nm03/lxy581/evaluate/amp_phase/form_vel_amp_pha_test_013.nc', lease_id='cafcc8297e764e90bef6e981e394009c'. This can happen if the Lock or Semaphore timed out before.
2026-04-21 11:10:08,350 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/nm03/lxy581/evaluate/amp_phase/form_vel_amp_pha_test_013.nc', lease_id='b54ab4a6f9c1485d8b84e729d6e9afdc'. This can happen if the Lock or Semaphore timed out before.
2026-04-21 11:15:19,540 - distributed.semaphore - WARNING - Tried to release Lock or Sem

In [3]:
time_len = 236
freq = 2 * np.pi / 12.4206014

In [4]:
chunks = 200
coords = ("xh", "xq", "yh", "yq")
coord_chunks = {v: chunks for v in coords}

In [5]:
weights_base = Path("/g/data/nm03/lxy581/global_drag_coeff/")
weights_h2u = weights_base / "8km_h_to_8km_u_weights.nc"
weights_h2v = weights_base / "8km_h_to_8km_v_weights.nc"

In [6]:
output_base = Path("/scratch/nm03/lxy581/mom6/archive/tides_008_global_sigma_SAL_2layer_x02/output013")
output_interior = output_base / 'ocean_interior.nc'
output_static = output_base / "ocean_static.nc"

In [7]:
def f(t, amp, phase, err):
    return amp * np.cos(freq * t - phase)

In [8]:
data = xr.open_dataset(output_interior, chunks=coord_chunks | {"time": 1})
static = xr.open_dataset(output_static, chunks=coord_chunks)

In [9]:
grid_h = xr.Dataset({"lat": static.geolat, "lon": static.geolon})
grid_u = xr.Dataset({"lat": static.geolat_u, "lon": static.geolon_u})
grid_v = xr.Dataset({"lat": static.geolat_v, "lon": static.geolon_v})

In [10]:
regrid_h2u = xe.Regridder(grid_h, grid_u, "bilinear", extrap_method="inverse_dist", reuse_weights=True, 
                          weights=weights_h2u)

In [11]:
regrid_h2v = xe.Regridder(grid_h, grid_v, "bilinear", extrap_method="inverse_dist", reuse_weights=True, 
                          weights=weights_h2v)

In [12]:
e = data.e.isel(time=slice(None, time_len)).squeeze()

In [13]:
e_top = e.isel(zi=0)
e_mid = e.isel(zi=1)
e_bot = e.isel(zi=2)

In [14]:
start_time = time.time()

In [15]:
# Regrid once each
e_top_u = regrid_h2u(e_top).persist()
e_mid_u = regrid_h2u(e_mid).persist()
e_bot_u = regrid_h2u(e_bot).persist()

e_top_v = regrid_h2v(e_top).persist()
e_mid_v = regrid_h2v(e_mid).persist()
e_bot_v = regrid_h2v(e_bot).persist()

In [16]:
end_time = time.time()
exe_time = float(end_time - start_time)
print("Execution time: %.1f seconds! \n" % exe_time)

Execution time: 3104.7 seconds! 



#### check shape

In [17]:
print(e_top_u.shape, e_top_v.shape)
print(e_mid_u.shape, e_mid_v.shape)
print(e_bot_u.shape, e_bot_v.shape)

(236, 3270, 4321) (236, 3271, 4320)
(236, 3270, 4321) (236, 3271, 4320)
(236, 3270, 4321) (236, 3271, 4320)


#### barotropic velocities (thickness-weighted)

In [18]:
start_time = time.time()

In [19]:
uo = data.uo.isel(time=slice(None, time_len)).squeeze()
vo = data.vo.isel(time=slice(None, time_len)).squeeze()

In [20]:
uh_top = uo.isel(zl=0) * (e_mid_u - e_top_u)
uh_bot = uo.isel(zl=1) * (e_bot_u - e_mid_u)

vh_top = vo.isel(zl=0) * (e_mid_v - e_top_v)
vh_bot = vo.isel(zl=1) * (e_bot_v - e_mid_v)

In [21]:
end_time = time.time()
exe_time = float(end_time - start_time)
print("Execution time: %.1f seconds! \n" % exe_time)

Execution time: 3.8 seconds! 



In [22]:
# -1 means: use the full time dimension as one chunk.
bt_u = ((uh_top + uh_bot) / (e_bot_u - e_top_u)).chunk({"time": -1}).persist()
bt_v = ((vh_top + vh_bot) / (e_bot_v - e_top_v)).chunk({"time": -1}).persist()

In [23]:
print(bt_u.shape)
print(bt_v.shape)

(236, 3270, 4321)
(236, 3271, 4320)


In [24]:
new_time = np.arange(time_len) + 130 * 24
bt_u = bt_u.assign_coords(time=new_time)
bt_v = bt_v.assign_coords(time=new_time)

In [25]:
fit_x = bt_u.curvefit("time", f).persist()
fit_y = bt_v.curvefit("time", f).persist()

coef_x = fit_x.curvefit_coefficients
coef_y = fit_y.curvefit_coefficients

In [26]:
amp_x = coef_x.isel(param=0)
pha_x = coef_x.isel(param=1)

amp_y = coef_y.isel(param=0)
pha_y = coef_y.isel(param=1)

In [27]:
amp_x_final = abs(amp_x)
pha_x_adjusted = xr.where(amp_x < 0, pha_x + np.pi, pha_x)
pha_x_adjusted = pha_x_adjusted % (2 * np.pi)
pha_x_final = xr.where(pha_x_adjusted > np.pi, pha_x_adjusted - 2*np.pi, pha_x_adjusted)

amp_y_final = abs(amp_y)
pha_y_adjusted = xr.where(amp_y < 0, pha_y + np.pi, pha_y)
pha_y_adjusted = pha_y_adjusted % (2 * np.pi)
pha_y_final = xr.where(pha_y_adjusted > np.pi, pha_y_adjusted - 2*np.pi, pha_y_adjusted)

In [28]:
start_time = time.time()

In [29]:
est_data = xr.Dataset(data_vars={"amp_x": amp_x_final,
                                 "phase_x": pha_x_final,
                                 "amp_y": amp_y_final,
                                 "phase_y": pha_y_final},
                      coords={"yh": data.yh, "xq": data.xq, "xh": data.xh, "yq": data.yq,})

est_data.to_netcdf("/g/data/nm03/lxy581/evaluate/amp_phase/form_vel_amp_pha_test_013.nc")

2026-04-21 11:23:52,352 - distributed.semaphore - ERROR - Release failed for id=0793caf57bce4913acaa136396e18d5d, lease_id=d1031020586547fc85ad17ea298a0d64, name=/g/data/nm03/lxy581/evaluate/amp_phase/form_vel_amp_pha_test_013.nc. Cluster network might be unstable?
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/utils.py", line 1928, in wait_for
    return await fut
           ^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/comm/tcp.py", line 547, in connect
    stream = await self.client.connect(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/tornado/tcpclient.py", line 279, in connect
    af, addr, stream = await connector.start(connect_timeout=timeout)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
asyncio.exceptions.Ca

ValueError: Missing dependency ('transpose-80be21ca3becad9492bb58cd9021e7ba', 4, 2, 0) for dependents {('getitem-d306506c4d1af6b2ca861591023e7bed', 4, 2)}

In [ ]:
end_time = time.time()
exe_time = float(end_time - start_time)
print("Execution time: %.1f seconds! \n" % exe_time)